In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from weaver.utils.dataset import SimpleIterDataset
from weaver.utils.import_tools import import_module

import sys

# test import onnx and test if onnxscript exists
try:
    import onnx
    from onnxscript import script
    print("ONNX and ONNXScript are available.")
except ImportError as e:
    print("ONNX or ONNXScript is not available. Please install them to use ONNX features.")
    print(str(e))

ONNX and ONNXScript are available.


In [14]:
# for comparison, also load the original model and check the outputs here
_mod_check_ = import_module('/afs/cern.ch/work/s/sewuchte/private/ScoutingProd/TrainTuples/TrainingCode/weaver-core/weaver/networks/example_ParticleTransformer2024PlusTagger_unified2SV.py', '_mod_check_')

data_config_check_ = f'/afs/cern.ch/work/s/sewuchte/private/ScoutingProd/TrainTuples/TrainingCode/weaver-core/weaver/data_new/incl_v10_ak15_full/ak15_MD_incl_v10beta4_ul_full_manual.yaml'
data_config_check_ = SimpleIterDataset({"_": []}, data_config_check_, for_training=False).config

# not using selected nodes
version = 'beta4p1'
selected_indices = None
# finetune_target_indices = [16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,7,3,0,1,2,3,369,370,371,372,373]
finetune_target_indices = None
num_nodes, num_cls_nodes = 750, 374

model_kwargs_check_ = dict(
    num_nodes=num_nodes,
    num_cls_nodes=num_cls_nodes,
    use_swiglu_config=True,
    use_pair_norm_config=True,
    embed_dims=[256, 1024, 256],
    fc_params=[(2048, 0.1)],
    pair_embed_dims=[64, 64, 64],
    num_heads=16,
    num_layers=10,
    reg_kw={'gamma': 5., 'composed_split_reg': [True, False], 'as_resid_of': [0]}, # beta 4.1's config
    use_amp=True,
    # aux fc
    # finetune_kw={
    #     'mode': 'cls',
    #     'target_inds': finetune_target_indices,
    #     'num_ft_nodes': 28,
    #     'fc_params': [(256,0.),(256,0.)],
    # },
    # finetune_kw=None,
    # finetune_kw={
    #     'mode': 'None',
    #     # 'mode': 'cls',
    #     # 'target_inds': finetune_target_indices,
    #     'target_inds': None,
    #     # 'num_ft_nodes': 28,
    #     # 'num_ft_nodes': 0,
    #     'num_ft_nodes': None,
    #     # 'fc_params': [(256,0.),(256,0.)],
    #     'fc_params': None,
    #     # 'input_highlevel_dim': 0,
    #     'input_highlevel_dim': None,
    # },
    export_params = {
        "apply_softmax": True,
        "concat_hid": False,
        "num_cls": 374,
    },
    # GloParT
    # num_output_nodes=22,
    # num_output_nodes=19,
    # version=version,
    # selected_indices=selected_indices,
    # misc
    for_inference=True, # important!
    )

model_check_, model_info_check_ = _mod_check_.get_model(data_config_check_, **model_kwargs_check_)
model_check_ = model_check_.to('cpu')

# aux_model_state = torch.load('model/ak15_MD_incl_v10beta4_ul_full_manual.nlayer10.vispart_as_resid.ddp4-bs640-lr1p2e-3.nepoch100.testrun/net_epoch-99_state.pt', map_location='cpu')

# new_state = {}
# for key in aux_model_state.keys():
    # print(key, aux_model_state[key].shape)
    # if key in ['main.part.fc.1.weight', 'main.part.fc.1.bias'] and selected_indices is not None:
    #     new_state[key] = aux_model_state[key][selected_indices]
    # else:
    #     new_state[key] = aux_model_state[key]

# missing_keys, unexpected_keys = model_check_.load_state_dict(new_state, strict=False)
# missing_keys, unexpected_keys = model_check_.load_state_dict(aux_model_state, strict=False)

ckpt_path = 'model/ak15_MD_incl_v10beta4_ul_full_manual.nlayer10.vispart_as_resid.ddp4-bs640-lr1p2e-3.nepoch100.testrun/net_epoch-99_state.pt'
aux = torch.load(ckpt_path, map_location='cpu')

# Inspect top-level keys
if isinstance(aux, dict):
    print("Checkpoint top-level keys:", list(aux.keys())[:20])
# Extract candidate state_dict
if isinstance(aux, dict) and 'state_dict' in aux:
    sd = aux['state_dict']
elif isinstance(aux, dict) and 'model' in aux:
    sd = aux['model']
else:
    sd = aux

# Strip DDP 'module.' prefix if present
new_sd = {}
for k, v in sd.items():
    new_k = k[7:] if k.startswith('module.') else k
    new_sd[new_k] = v

missing_keys, unexpected_keys = model_check_.load_state_dict(new_sd, strict=False)
print('Missing keys (len):', len(missing_keys))
print('Unexpected keys (len):', len(unexpected_keys))
if len(missing_keys) <= 40:
    print('Missing keys sample:', missing_keys[:20])
if len(unexpected_keys) <= 20:
    print('Unexpected keys sample:', unexpected_keys[:20])

# Quick parameter sanity-check (print mean of first few params)
for name, p in model_check_.named_parameters():
    print('param:', name, 'mean:', float(p.data.mean()))
    break

# Make deterministic where possible, then do forward
import numpy as np
torch.manual_seed(0)
np.random.seed(0)
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

arrs_check_ = [torch.ones(1, 30, 90), torch.ones(1, 4, 90), torch.ones(1, 1, 90), torch.ones(1, 7, 60), torch.ones(1, 4, 60), torch.ones(1, 1, 60), torch.ones(1, 11, 10), torch.ones(1, 4, 10), torch.ones(1, 1, 10)]


model_check_.eval()
with torch.no_grad():
    output_check_ = model_check_(arrs_check_[0], arrs_check_[1], arrs_check_[2],
                                 arrs_check_[3], arrs_check_[4], arrs_check_[5],
                                 arrs_check_[6], arrs_check_[7], arrs_check_[8])
print("Output shape:", output_check_.shape)
print("Score output values:", output_check_[0, :5])


# test one forward pass with dummy input with shapes 'input_shapes': {'cpf_features': (1, 30, 90), 'cpf_vectors': (1, 4, 90), 'cpf_mask': (1, 1, 90), 'npf_features': (1, 7, 60), 'npf_vectors': (1, 4, 60), 'npf_mask': (1, 1, 60), 'sv_features': (1, 11, 10), 'sv_vectors': (1, 4, 10), 'sv_mask': (1, 1, 10)}
# dummy_input_check_ = {
#     'cpf_features': torch.ones(1, 30, 90),
#     'cpf_vectors': torch.ones(1, 4, 90),
#     'cpf_mask': torch.ones(1, 1, 90),
#     'npf_features': torch.ones(1, 7, 60),
#     'npf_vectors': torch.ones(1, 4, 60),
#     'npf_mask': torch.ones(1, 1, 60),
#     'sv_features': torch.ones(1, 11, 10),
#     'sv_vectors': torch.ones(1, 4, 10),
#     'sv_mask': torch.ones(1, 1, 10),
#     # 'dummy_highlevel': torch.randn(1, 0), # for finetune_kw's input_highlevel_dim
# }

# model_check_.eval()
# with torch.no_grad():
#     output_check_ =    model_check_(arrs_check_[0], arrs_check_[1], arrs_check_[2], arrs_check_[3], arrs_check_[4], arrs_check_[5], arrs_check_[6], arrs_check_[7], arrs_check_[8])
# print("Output shape:", output_check_.shape)


# print the Hbb score and massCorrResonance output values
print("Score output values:", output_check_[0, :5])
print("massCorrResonance output value:", output_check_[0, 374])
print("massCorrGeneric output value:", output_check_[0, 375])

in get_model, kwargs: {'num_nodes': 750, 'num_cls_nodes': 374, 'use_swiglu_config': True, 'use_pair_norm_config': True, 'embed_dims': [256, 1024, 256], 'fc_params': [(2048, 0.1)], 'pair_embed_dims': [64, 64, 64], 'num_heads': 16, 'num_layers': 10, 'reg_kw': {'gamma': 5.0, 'composed_split_reg': [True, False], 'as_resid_of': [0]}, 'use_amp': True, 'export_params': {'apply_softmax': True, 'concat_hid': False, 'num_cls': 374}, 'for_inference': True}
Model config after update with kwargs: {'input_dims': (30, 7, 11), 'share_embed': False, 'num_classes': 750, 'pair_input_type': 'pp', 'pair_input_dim': 6, 'pair_extra_dim': 0, 'use_pair_norm': True, 'remove_self_pair': False, 'use_pre_activation_pair': True, 'embed_dims': [256, 1024, 256], 'pair_embed_dims': [64, 64, 64], 'num_heads': 16, 'num_layers': 10, 'num_cls_layers': 2, 'block_params': {'scale_attn_mask': True, 'scale_attn': False, 'scale_fc': False, 'scale_heads': False, 'scale_resids': False, 'activation': 'swiglu'}, 'cls_block_params'

/tmp/ipykernel_3229108/2046662069.py:78: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  aux = torch.load(ckpt_path, map_location='cpu')


Checkpoint top-level keys: ['input_embeds.0.input_bn.weight', 'input_embeds.0.input_bn.bias', 'input_embeds.0.input_bn.running_mean', 'input_embeds.0.input_bn.running_var', 'input_embeds.0.input_bn.num_batches_tracked', 'input_embeds.0.embed.0.weight', 'input_embeds.0.embed.0.bias', 'input_embeds.0.embed.1.weight', 'input_embeds.0.embed.1.bias', 'input_embeds.0.embed.3.weight', 'input_embeds.0.embed.3.bias', 'input_embeds.0.embed.4.weight', 'input_embeds.0.embed.4.bias', 'input_embeds.0.embed.6.weight', 'input_embeds.0.embed.6.bias', 'input_embeds.0.embed.7.weight', 'input_embeds.0.embed.7.bias', 'input_embeds.1.input_bn.weight', 'input_embeds.1.input_bn.bias', 'input_embeds.1.input_bn.running_mean']
Missing keys (len): 0
Unexpected keys (len): 0
Missing keys sample: []
Unexpected keys sample: []
param: input_embeds.0.input_bn.weight mean: 0.7436046004295349
args len: 9, expected: 9
num_cls: 374
output with length 750 before softmax:

output_cls:
 tensor([[-11.2777, -15.6184, -21.5643,

In [18]:

sys.path.append('/afs/cern.ch/work/s/sewuchte/private/ScoutingProd/TrainTuples/TrainingCode/weaver-core/weaver')

_mod = import_module('/afs/cern.ch/work/s/sewuchte/private/ScoutingProd/TrainTuples/TrainingCode/weaver-core/weaver/networks/stage3/example_GloParT3_exporter_final_copy2.py', '_mod')

data_config = f'/afs/cern.ch/work/s/sewuchte/private/ScoutingProd/TrainTuples/TrainingCode/weaver-core/weaver/data_new/incl_v10_ak15_full/ak15_MD_incl_v10beta4_ul_full_manual.yaml'
data_config = SimpleIterDataset({"_": []}, data_config, for_training=False).config

# not using selected nodes
version = 'beta4p1'
selected_indices = None
# finetune_target_indices = [16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,7,3,0,1,2,3,369,370,371,372,373]
finetune_target_indices = None
num_nodes, num_cls_nodes = 750, 374

# # ## use selected nodes
# version = 'beta4p1:selected_indices'
# # selected_indices = [0, 1, 2, 3, 7, 8, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 369, 370, 371, 372, 373, 375, 376, 377, 378, 379, 383, 384]
# # num_nodes, num_cls_nodes = 61, 54
# selected_indices = list(range(374)) + [375, 376, 377, 378, 379, 383, 384]
# num_nodes, num_cls_nodes = 381, 374
# finetune_target_indices = [selected_indices.index(i) for i in finetune_target_indices]

# == args for train_GloParT_v3beta4.sh ==
# -o num_nodes 750 -o num_cls_nodes 374 -o use_swiglu_config True -o use_pair_norm_config True \
# -o fc_params [(2048,0.1)] -o embed_dims [256,1024,256] -o pair_embed_dims [64,64,64] -o num_heads 16 -o num_layers 12 \
# -o reg_kw {'gamma':5.,'composed_split_reg':[True,False],'as_resid_of':[0]} \
# --use-amp --batch-size 512 --start-lr 7e-4 --num-epochs 100 --optimizer ranger \

# modelftopts="-o finetune_kw {'mode':'cls','target_inds':[16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,7,3,0,1,2,3,369,370,371,372,373],'num_ft_nodes':28,'fc_params':[(256,0.),(256,0.)]} \
# --load-model-weights finetune_stage3beta4p1 --freeze-model-weights main* \
# --train-mode cls --num-epochs 30 "
# modelopts="-o num_layers 10 -o reg_kw {'gamma':5.,'composed_split_reg':[True,False],'as_resid_of':[1]} " # beta 4.1's config

model_kwargs = dict(
    num_nodes=num_nodes,
    num_cls_nodes=num_cls_nodes,
    use_swiglu_config=True,
    use_pair_norm_config=True,
    embed_dims=[256, 1024, 256],
    fc_params=[(2048, 0.1)],
    pair_embed_dims=[64, 64, 64],
    num_heads=16,
    # num_layers=12,
    num_layers=10,
    reg_kw={'gamma': 5., 'composed_split_reg': [True, False], 'as_resid_of': [0]}, # beta 4.1's config
    use_amp=True,
    # aux fc
    # finetune_kw={
    #     'mode': 'cls',
    #     'target_inds': finetune_target_indices,
    #     'num_ft_nodes': 28,
    #     'fc_params': [(256,0.),(256,0.)],
    # },
    # finetune_kw={
    #     'mode': 'None',
    #     # 'mode': 'cls',
    #     # 'target_inds': finetune_target_indices,
    #     'target_inds': None,
    #     'target_inds_opt': None,
    #     # 'num_ft_nodes': 28,
    #     # 'num_ft_nodes': 0,
    #     'num_ft_nodes': None,
    #     # 'fc_params': [(256,0.),(256,0.)],
    #     'fc_params': None,
    #     # 'input_highlevel_dim': 0,
    #     'input_highlevel_dim': None,
    # },
    # GloParT
    # num_output_nodes=22,
    # num_output_nodes=19,
    num_output_nodes=20, #try this
    version=version,
    # selected_indices=selected_indices,
    # misc
    for_inference=True, # important!
    # freeze_main_params=True, # important!
    export_params = {
        "apply_softmax": True,
        # "apply_softmax": False,
        "concat_hid": False,
        "num_cls": 374,
    },
    )

model, model_info = _mod.get_model(data_config, **model_kwargs)
model = model.to('cpu')

# aux_model_state = torch.load('model/ak15_MD_incl_v10beta4_ul_full_manual.nlayer10.vispart_as_resid.ddp4-bs640-lr1p2e-3.nepoch100.testrun/net_epoch-99_state.pt', map_location='cpu')

# new_state = {}
# for key in aux_model_state.keys():
#     # print(key, aux_model_state[key].shape)
#     if key in ['main.part.fc.1.weight', 'main.part.fc.1.bias'] and selected_indices is not None:
#         new_state[key] = aux_model_state[key][selected_indices]
#     else:
#         new_state[key] = aux_model_state[key]

# missing_keys, unexpected_keys = model.load_state_dict(new_state, strict=False)



ckpt_path = 'model/ak15_MD_incl_v10beta4_ul_full_manual.nlayer10.vispart_as_resid.ddp4-bs640-lr1p2e-3.nepoch100.testrun/net_epoch-99_state.pt'
aux = torch.load(ckpt_path, map_location='cpu')

# Inspect top-level keys
if isinstance(aux, dict):
    print("Checkpoint top-level keys:", list(aux.keys())[:20])
# Extract candidate state_dict
if isinstance(aux, dict) and 'state_dict' in aux:
    sd = aux['state_dict']
elif isinstance(aux, dict) and 'model' in aux:
    sd = aux['model']
else:
    sd = aux

# Strip DDP 'module.' prefix if present
new_sd = {}
for k, v in sd.items():
    new_k = k[7:] if k.startswith('module.') else k
    new_sd[new_k] = v

missing_keys, unexpected_keys = model.load_state_dict(new_sd, strict=False)
print('Missing keys (len):', len(missing_keys))
print('Unexpected keys (len):', len(unexpected_keys))
# if len(missing_keys) <= 40:
print('Missing keys sample:', missing_keys[:20])
# if len(unexpected_keys) <= 20:
print('Unexpected keys sample:', unexpected_keys[:20])

# now loop over the keys, and for every key that is missing but exists as the same name without main.* in the unexpected keys, copy the value from the unexpected key to the missing key in the model's state dict, and remove that key from the unexpected keys list. This is to handle the case where the checkpoint was saved with DDP and has 'module.' prefixes, but the current model does not have those prefixes.
model_sd = model.state_dict()
for missing_key in missing_keys:
    if missing_key.startswith('main.'):
        alt_key = missing_key[5:] # remove 'main.' prefix
        if alt_key in unexpected_keys:
            print(f"Copying value from unexpected key '{alt_key}' to missing key '{missing_key}'")
            model_sd[missing_key] = new_sd[alt_key]
            unexpected_keys.remove(alt_key)
# recheck the missing and unexpected keys after this copying
missing_keys, unexpected_keys = model.load_state_dict(model_sd, strict=False)
print('After copying, missing keys (len):', len(missing_keys))
print('After copying, unexpected keys (len):', len(unexpected_keys))
print('After copying, missing keys sample:', missing_keys[:20])
print('After copying, unexpected keys sample:', unexpected_keys[:20])


# print the keys and shapes of the model
# for k, v in model.state_dict().items():
#     print(k, v.shape)
print(model_info)


# test one forward pass with dummy input with shapes 'input_shapes': {'cpf_features': (1, 30, 90), 'cpf_vectors': (1, 4, 90), 'cpf_mask': (1, 1, 90), 'npf_features': (1, 7, 60), 'npf_vectors': (1, 4, 60), 'npf_mask': (1, 1, 60), 'sv_features': (1, 11, 10), 'sv_vectors': (1, 4, 10), 'sv_mask': (1, 1, 10)}
# dummy_input = {
#     'cpf_features': torch.ones(1, 30, 90),
#     'cpf_vectors': torch.ones(1, 4, 90),
#     'cpf_mask': torch.ones(1, 1, 90),
#     'npf_features': torch.ones(1, 7, 60),
#     'npf_vectors': torch.ones(1, 4, 60),
#     'npf_mask': torch.ones(1, 1, 60),
#     'sv_features': torch.ones(1, 11, 10),
#     'sv_vectors': torch.ones(1, 4, 10),
#     'sv_mask': torch.ones(1, 1, 10),
#     # 'dummy_highlevel': torch.randn(1, 0), # for finetune_kw's input_highlevel_dim
# }
# dummy_input = dummy_input_check_
arrs = [torch.ones(1, 30, 90), torch.ones(1, 4, 90), torch.ones(1, 1, 90), torch.ones(1, 7, 60), torch.ones(1, 4, 60), torch.ones(1, 1, 60), torch.ones(1, 11, 10), torch.ones(1, 4, 10), torch.ones(1, 1, 10)]

model.eval()
with torch.no_grad():
    # output = model(arrs[0], arrs[1], arrs[2], arrs[3], arrs[4], arrs[5], arrs[6], arrs[7], arrs[8], dummy_input['dummy_highlevel'])
    output = model(arrs[0], arrs[1], arrs[2], arrs[3], arrs[4], arrs[5], arrs[6], arrs[7], arrs[8])
print("Output shape:", output.shape)


# print the Hbb score and massCorrResonance output values
print("Score output values:", output[0, :17])
print("massCorrX2p output value:", output[0, 17])
print("massCorrGeneric output value:", output[0, 18])
print("massCorrResonance output value:", output[0, 19])

in get_model, kwargs: {'num_nodes': 750, 'num_cls_nodes': 374, 'use_swiglu_config': True, 'use_pair_norm_config': True, 'embed_dims': [256, 1024, 256], 'fc_params': [(2048, 0.1)], 'pair_embed_dims': [64, 64, 64], 'num_heads': 16, 'num_layers': 10, 'reg_kw': {'gamma': 5.0, 'composed_split_reg': [True, False], 'as_resid_of': [0]}, 'use_amp': True, 'num_output_nodes': 20, 'version': 'beta4p1', 'for_inference': True, 'export_params': {'apply_softmax': True, 'concat_hid': False, 'num_cls': 374}}
Model config after update with kwargs: {'input_dims': (30, 7, 11), 'share_embed': False, 'num_classes': 750, 'pair_input_type': 'pp', 'pair_input_dim': 6, 'pair_extra_dim': 0, 'use_pair_norm': True, 'remove_self_pair': False, 'use_pre_activation_pair': True, 'embed_dims': [256, 1024, 256], 'pair_embed_dims': [64, 64, 64], 'num_heads': 16, 'num_layers': 10, 'num_cls_layers': 2, 'block_params': {'scale_attn_mask': True, 'scale_attn': False, 'scale_fc': False, 'scale_heads': False, 'scale_resids': Fals

/tmp/ipykernel_3229108/471241796.py:103: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  aux = torch.load(ckpt_path, map_location='cpu')


Checkpoint top-level keys: ['input_embeds.0.input_bn.weight', 'input_embeds.0.input_bn.bias', 'input_embeds.0.input_bn.running_mean', 'input_embeds.0.input_bn.running_var', 'input_embeds.0.input_bn.num_batches_tracked', 'input_embeds.0.embed.0.weight', 'input_embeds.0.embed.0.bias', 'input_embeds.0.embed.1.weight', 'input_embeds.0.embed.1.bias', 'input_embeds.0.embed.3.weight', 'input_embeds.0.embed.3.bias', 'input_embeds.0.embed.4.weight', 'input_embeds.0.embed.4.bias', 'input_embeds.0.embed.6.weight', 'input_embeds.0.embed.6.bias', 'input_embeds.0.embed.7.weight', 'input_embeds.0.embed.7.bias', 'input_embeds.1.input_bn.weight', 'input_embeds.1.input_bn.bias', 'input_embeds.1.input_bn.running_mean']
Missing keys (len): 261
Unexpected keys (len): 269
Missing keys sample: ['main.input_embeds.0.input_bn.weight', 'main.input_embeds.0.input_bn.bias', 'main.input_embeds.0.input_bn.running_mean', 'main.input_embeds.0.input_bn.running_var', 'main.input_embeds.0.embed.0.weight', 'main.input_em

In [ ]:
# aux_model_state = torch.load('/home/olympus/licq/hww/incl-train/weaver-core/weaver/model/ak8_MD_inclv10_nonmd_witheta_manual.lr2e-2.fixdataloader.origmodel.ak8_MD_inclv10beta4_ul_manual.nlayer10.vispart_as_resid.ddp4-bs640-lr1p2e-3.nepoch100.farm221/net_best_epoch_state.pt', map_location='cpu')
# aux_model_state = torch.load('model/ak15_MD_incl_v10beta4_ul_full_manual.nlayer10.vispart_as_resid.ddp4-bs640-lr1p2e-3.nepoch100.testrun/net_epoch-99_state.pt', map_location='cpu')

# new_state = {}
# for key in aux_model_state.keys():
#     print(key, aux_model_state[key].shape)
#     if key in ['main.part.fc.1.weight', 'main.part.fc.1.bias'] and selected_indices is not None:
#         new_state[key] = aux_model_state[key][selected_indices]
#     else:
#         new_state[key] = aux_model_state[key]

# missing_keys, unexpected_keys = model.load_state_dict(new_state, strict=False)
# print('Model initialized with weights.. Missing: %s\n ... Unexpected: %s' % (missing_keys, unexpected_keys))

In [19]:
model.eval()
inputs = tuple(
    torch.ones(model_info['input_shapes'][k], dtype=torch.float32) for k in model_info['input_names'])
torch.onnx.export(model, inputs, 'model_opset11.onnx',
                    input_names=model_info['input_names'],
                    output_names=model_info['output_names'],
                    dynamic_axes=model_info.get('dynamic_axes', None),
                    opset_version=14) # 11 for 10_6 (using pytorch1), 14 for Run 3

print (model_info['output_names'])


args len: 9, expected: 9


/afs/cern.ch/work/s/sewuchte/private/ScoutingProd/TrainTuples/TrainingCode/weaver-core/weaver/networks/stage3/../ParticleTransformer2024Plus.py:545: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert key_padding_mask.shape == (bsz, src_len), \
/afs/cern.ch/work/s/sewuchte/private/ScoutingProd/TrainTuples/TrainingCode/weaver-core/weaver/networks/stage3/../ParticleTransformer2024Plus.py:551: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert attn_mask.shape == (bsz, self.num_heads, tgt_len, src_len), \
/afs/cern.ch/work/s/sewuchte/private/ScoutingPro

num_cls: 374
output with length 750 before softmax:

output_cls:
 tensor([[-11.2776, -15.6183, -21.5643, -22.6771,  -9.8652,  -6.5083, -15.3514,
         -19.9657, -12.1014,  -6.3471, -16.8626, -16.6586,  -8.4577, -17.0951,
          -9.9557, -19.5243,  -9.7581, -14.8658,  -1.8830,  -9.2323, -10.3884,
          -7.2231,  -5.1386,  -5.5074,  -2.5629,  -7.0889, -14.7483, -13.8222,
          -7.9957,  -3.3958,  -8.0318,  -2.9109, -12.6400,  -4.9330,  -9.3846,
          -7.0792,  -6.7462,  -7.1751,   0.6712,   1.0413,  -1.2941,   1.3172,
          -0.5793,  -6.6538,  -6.3555,  -1.7940,  -2.3035,  -2.6018,  -1.6948,
          -4.2866, -10.6835, -15.8361, -16.5400,  -8.3387, -10.8587, -11.5939,
         -13.2138,  -9.9273, -11.7221, -12.4220, -14.3457,  -9.0763, -12.6458,
         -12.5247, -12.1820,  -8.4115, -10.8652, -12.1810, -18.2587,  -8.1523,
          -9.1896, -12.1409,  -4.7000, -13.6840,  -9.5765,  -8.5715,  -8.9644,
         -11.1042, -14.1837, -15.6627, -12.1789,  -9.6645, -14.99

In [21]:
# open the onnx model and print the input and output names and shapes
onnx_model = onnx.load('model_opset11.onnx')
print("ONNX model inputs:")
for input in onnx_model.graph.input:
    name = input.name
    shape = [dim.dim_value for dim in input.type.tensor_type.shape.dim]
    print(f"  {name}: {shape}")

# the outputs
print("ONNX model outputs:")
for output in onnx_model.graph.output:
    name = output.name
    shape = [dim.dim_value for dim in output.type.tensor_type.shape.dim]
    print(f"  {name}: {shape}")


dummy_input = {
    'cpf_features': torch.ones(1, 30, 90),
    'cpf_vectors': torch.ones(1, 4, 90),
    'cpf_mask': torch.ones(1, 1, 90),
    'npf_features': torch.ones(1, 7, 60),
    'npf_vectors': torch.ones(1, 4, 60),
    'npf_mask': torch.ones(1, 1, 60),
    'sv_features': torch.ones(1, 11, 10),
    'sv_vectors': torch.ones(1, 4, 10),
    'sv_mask': torch.ones(1, 1, 10),
    # 'dummy_highlevel': torch.randn(1, 0), # for finetune_kw's input_highlevel_dim
}

# use the dummy input from above to test the onnx model with onnxruntime
import onnxruntime as ort
ort_session = ort.InferenceSession('model_opset11.onnx')
onnx_inputs = {k: v.cpu().numpy() for k, v in dummy_input.items()}
onnx_inputs = {k: v for k, v in onnx_inputs.items() if k in model_info['input_names']}
dummy_output = ort_session.run(model_info['output_names'], onnx_inputs)[0]
print("ONNX model output shape:", dummy_output.shape)


# print the outputs
print("Output values:", dummy_output[0, :17])
print ("massCorrX2p output value:", dummy_output[0, 17])
print ("massCorrGeneric output value:", dummy_output[0, 18])
print ("massCorrResonance output value:", dummy_output[0, 19])

ONNX model inputs:
  cpf_features: [0, 30, 0]
  cpf_vectors: [0, 4, 0]
  cpf_mask: [0, 1, 0]
  npf_features: [0, 7, 0]
  npf_vectors: [0, 4, 0]
  npf_mask: [0, 1, 0]
  sv_features: [0, 11, 0]
  sv_vectors: [0, 4, 0]
  sv_mask: [0, 1, 0]
ONNX model outputs:
  output: [0, 20]
ONNX model output shape: (1, 20)
Output values: [9.0428472e-07 1.1780414e-08 3.9691821e-07 5.1080043e-11 2.6903360e-09
 3.3917161e-06 2.3705332e-10 1.6523196e-06 2.3321298e-05 3.3000694e-07
 8.4028607e-06 5.2510545e-04 1.1080973e-02 1.3989402e-01 2.0288655e-01
 4.0101279e-02 1.0183453e-02]
massCorrX2p output value: 1.0427585
massCorrGeneric output value: 0.60322046
massCorrResonance output value: 1.1341946
